In [1]:
import pandas as pd
import numpy as np
from os import listdir
import re

# Parsing

In [2]:
def CompileCSV(FOLDER : str):
    FILENAMES = [f'{FOLDER}/{file}' for file in listdir(FOLDER)[1:]]
    FILENAMES
    
    output = None
    
    for f in FILENAMES:
        if f[-4:] != '.csv':
            break
    
        df = None
    
        YEAR, SEASON = f.split('/')[-1].strip('.csv').split('_')
        #print(YEAR, SEASON)
    
        df = pd.read_csv(f).drop(columns='Unnamed: 0')
        df['Year'] = YEAR
        df['Season'] = SEASON
    
    
        if output is None:
            output = df.copy()
        else:
            output = pd.concat([output, df])

    return output

In [3]:
df_ids = CompileCSV('../Scraping/Season')
df_ids['MAL_id'] = df_ids.Link.str.extract(r'myanimelist.net/anime/(\d+)').astype(int)
df_ids.head()

,Title,Link,Year,Season,MAL_id
0,Shinseiki Evangelion,https://myanimelist.net/anime/30/Shinseiki_Eva...,1995,fall,30
1,Gokinjo Monogatari,https://myanimelist.net/anime/852/Gokinjo_Mono...,1995,fall,852
2,Bakuretsu Hunters,https://myanimelist.net/anime/495/Bakuretsu_Hu...,1995,fall,495
3,Kaitou Saint Tail,https://myanimelist.net/anime/1567/Kaitou_Sain...,1995,fall,1567
4,Shinpi no Sekai El-Hazard (TV),https://myanimelist.net/anime/116/Shinpi_no_Se...,1995,fall,116


In [4]:
df = CompileCSV('../Scraping/Detail/')
df_ids = df_ids.drop_duplicates('MAL_id')
df = pd.merge(df, df_ids[['Title', 'MAL_id']], on='Title', how='left')
df = df.set_index('MAL_id')

df = df.drop(columns=['Synonyms', 'Japanese', 'English', 'German', 'Spanish', 'French', 'Aired','Broadcast','Premiered'])
df.head(3).T

MAL_id,23273,23755,22535
Type,['TV'],['TV'],['TV']
Episodes,22,24,24
Status,Finished Airing,Finished Airing,Finished Airing
Producers,"['Aniplex', 'Dentsu', 'Kodansha', 'Fuji TV', '...","['Aniplex', 'Dentsu', 'Mainichi Broadcasting S...","['VAP', 'Nippon Television Network', 'Forecast..."
Licensors,['Aniplex of America'],['Funimation'],['Sentai Filmworks']
Studios,['A-1 Pictures'],['A-1 Pictures'],['Madhouse']
Source,['Manga'],['Manga'],['Manga']
Genres,"['Drama', 'Romance']","['Action', 'Adventure', 'Fantasy']","['Action', 'Horror', 'Sci-Fi', 'Suspense']"
Themes,"['Love Polygon', 'Music', 'School']",NaN,"['Gore', 'Psychological']"
Demographic,['Shounen'],['Shounen'],['Seinen']


In [5]:
def ExtractList(col : str):
    df[col] = df[col].str.findall(r"'([^']*)'")

for c in ['Producers', 'Licensors', 'Studios', 'Themes', 'Theme', 'Genres', 'Genre', 'Demographics', 'Demographic']:
    ExtractList(c)

In [6]:
def ExtractFirstItemInList(col : str):
    df[col] = df[col].str.extract(r"'([^']*)'")
    
for c in ['Source', 'Type']:
    ExtractFirstItemInList(c)

In [7]:
df['Episodes'] = df['Episodes'].replace('Unknown', np.nan).astype(float)

In [8]:
df['Is_Finished'] = df.Status == 'Finished Airing'
df = df.drop(columns=['Status'])

In [9]:
ExtractList('Related_Entries')
df['Related_Entries'] = df['Related_Entries'].apply(lambda x : len(x))

In [10]:
df['Synopsis'] = df['Synopsis'].str.replace(r'\[.*Written.*?\]|\(.*Source.*?\)|^No synopsis information has been added to this title. Help improve our database by adding a synopsis here.', '', regex=True)

In [11]:
df['Year'] = df['Year'].astype(int)
df['Members'] = df['Members'].replace(',','', regex=True).astype(int)
df['Favorites'] = df['Favorites'].replace(',','', regex=True).astype(int)

In [12]:
df['Score'] = df['Score'].replace('[]', np.nan).astype(float)

In [13]:
df['Scorers'] = df['Scorers'].replace(np.nan, 0).astype(int)

In [14]:
df['Ranked'] = df['Ranked'].replace('#', '', regex=True).astype(float)
df['Popularity'] = df['Popularity'].replace('#', '', regex=True).astype(int)

In [15]:
df['Themes'] = df['Theme'].combine_first(df['Themes'])
df['Genres'] = df['Genre'].combine_first(df['Genres'])
df['Demographics'] = df['Demographic'].combine_first(df['Demographics'])
df = df.drop(columns=['Theme', 'Genre', 'Demographic'])

In [16]:
def ExtractMinutes(time_str : str):
    hours_match = re.search(r'(\d+)\s*hr', time_str)
    minutes_match = re.search(r'(\d+)\s*min', time_str)

    minutes = 0
    if hours_match:
        minutes += int(hours_match.group(1)) * 60
    if minutes_match:
        minutes += int(minutes_match.group(1)) 

    return minutes

df['Duration'] = df['Duration'].apply(lambda x: ExtractMinutes(str(x)))

In [17]:
df.head().T

MAL_id,23273,23755,22535,22297,25013
Type,TV,TV,TV,TV,TV
Episodes,22.0,24.0,24.0,12.0,24.0
Producers,"[Aniplex, Dentsu, Kodansha, Fuji TV, Lawson HM...","[Aniplex, Dentsu, Mainichi Broadcasting System...","[VAP, Nippon Television Network, Forecast Comm...","[Aniplex, Notes, Studio Mausu]","[VAP, Hakusensha, AT-X, Delfi Sound, Marvelous..."
Licensors,[Aniplex of America],[Funimation],[Sentai Filmworks],[Aniplex of America],[Funimation]
Studios,[A-1 Pictures],[A-1 Pictures],[Madhouse],[ufotable],[Pierrot]
Source,Manga,Manga,Manga,NaN,Manga
Genres,"[Drama, Romance]","[Action, Adventure, Fantasy]","[Action, Horror, Sci-Fi, Suspense]","[Action, Fantasy]","[Adventure, Fantasy, Romance]"
Themes,"[Love Polygon, Music, School]",NaN,"[Gore, Psychological]",[Urban Fantasy],NaN
Duration,22,24,23,28,24
Rating,PG-13 - Teens 13 or older,PG-13 - Teens 13 or older,R - 17+ (violence & profanity),R - 17+ (violence & profanity),PG-13 - Teens 13 or older


In [18]:
df.dtypes

Type                object
Episodes           float64
Producers           object
Licensors           object
Studios             object
Source              object
Genres              object
Themes              object
Duration             int64
Rating              object
Score              float64
Ranked             float64
Popularity           int64
Members              int64
Favorites            int64
Scorers              int64
Synopsis            object
Related_Entries      int64
Title               object
Demographics        object
Year                 int64
Season              object
Is_Finished           bool
dtype: object

In [19]:
df.to_csv('Cleaned.csv')

# Utility Matrix

In [20]:
def OneHotList(input_df, col, prefix):
    output = input_df[[col]]

    output = output.fillna('FILLNA')
    
    for i in output[col].explode().unique():
        output[f'{prefix}{i}'] = output[col].apply(lambda x: 1 if i in x else 0)

    if f'{prefix}FILLNA' in output.columns:
        output = output.drop(columns=[f'{prefix}FILLNA'])

    output = output.drop(columns=[col])
    return output

In [21]:
USM = df[['Title', 'Is_Finished']].copy()
USM['Is_Finished'] = USM['Is_Finished'].astype(np.int8)

items = [('Genres', 'Genre_'),
         ('Demographics', 'Demo_'),
         ('Themes', 'Theme_'),
         ('Studios', 'Studio_'),
         ('Producers', 'Producer_'),
         ('Licensors', 'Licensor_'),
        ]

for i in items:      
    USM = pd.concat(axis=1, objs= [USM, OneHotList(df, i[0], i[1])])

C:\Users\Admin\AppData\Local\Temp\ipykernel_19704\2727251450.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f'{prefix}{i}'] = output[col].apply(lambda x: 1 if i in x else 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_19704\2727251450.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f'{prefix}{i}'] = output[col].apply(lambda x: 1 if i in x else 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_19704\2727251450.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

In [22]:
df.loc[40776]

Type                                                              TV
Episodes                                                        12.0
Producers          [Dentsu, Mainichi Broadcasting System, Movic, ...
Licensors                                         [Sentai Filmworks]
Studios                                             [Production I.G]
Source                                                         Manga
Genres                                                      [Sports]
Themes                                         [School, Team Sports]
Duration                                                          23
Rating                                     PG-13 - Teens 13 or older
Score                                                           8.55
Ranked                                                         124.0
Popularity                                                       219
Members                                                       863774
Favorites                         

In [23]:
items = [
    ['Type', 'Type'],
    ['Rating', 'Rating'],
    ['Source', 'Source'],
]

for i in items:      
    USM = pd.concat(axis=1, objs= [USM, 
                                   pd.get_dummies(df[i[0]], prefix=i[1]).astype(np.int8)])

In [24]:
USM = pd.concat([USM, df[['Score', 'Members', 'Favorites', 'Related_Entries', 'Year', 'Episodes']]], axis=1)

In [25]:
USM

,Title,Is_Finished,Genre_Drama,Genre_Romance,Genre_Action,Genre_Adventure,Genre_Fantasy,Genre_Horror,Genre_Sci-Fi,Genre_Suspense,...,Source_4-koma manga,Source_Light novel,Source_Manga,Source_Web manga,Score,Members,Favorites,Related_Entries,Year,Episodes
MAL_id,,,,,,,,,,,,,,,,,,,,,
23273,Shigatsu wa Kimi no Uso,1,1,1,0,0,0,0,0,0,...,0,0,1,0,8.64,2297782,87444,2,2014,22.0
23755,Nanatsu no Taizai,1,0,0,1,1,1,0,0,0,...,0,0,1,0,7.62,2124863,19309,2,2014,24.0
22535,Kiseijuu: Sei no Kakuritsu,1,0,0,1,0,0,1,1,1,...,0,0,1,0,8.32,1946754,35687,2,2014,24.0
22297,Fate/stay night: Unlimited Blade Works,1,0,0,1,0,1,0,0,0,...,0,0,0,0,8.18,1109905,16907,4,2014,12.0
25013,Akatsuki no Yona,1,0,1,0,1,1,0,0,0,...,0,0,1,0,8.03,891045,16955,2,2014,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58606,Girls & Panzer: Saishuushou Part 4 Specials,1,0,0,0,0,0,0,0,0,...,0,0,0,0,7.29,2956,2,0,2024,2.0
58006,Science SARU x MBS Original Short Anime Daisak...,1,0,0,0,0,0,0,0,0,...,0,0,0,0,6.06,538,0,0,2024,4.0
58229,Harutsugeuo to Fuuraibou,1,0,0,0,1,0,0,0,0,...,0,0,0,0,NaN,232,0,0,2024,1.0


In [26]:
USM.to_csv('USM.csv')